# 6. Paper numbers

Runs on CPU in a few minutes, after notebooks 4 and 5.

Computes the in distribution against leave one scanner out AUCs (Table S5) and the intervals for the interventions in Table 3, then stacks the result tables in `paper_results` into one file, `LEDGER_all_results.csv`. It skips files with more than 400 rows, so `G_label_budget.csv` is not in the ledger.

**Step 1.** CELL 19: the remaining numbers (Table S5 and the intervals for the interventions in Table 3) and the ledger of the result tables. A comment in the code says "as in Table 4"; that table is Table S5 in the final supplement.

In [ ]:
# CELL 19 (CPU, ~3 min, any notebook): the last numbers the paper text needs, plus one ledger file of every result table
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, glob, os, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"; DG = f"{CKPT}/lifs_dg"
CENTERS = ["A", "B1", "B2"]

# (a) in-distribution AUC, per scanner and pooled: deep (3 seeds), deep (seed 42), forest (as CELL K), equal fusion (3 seeds)
ID3 = pd.read_csv(f"{RES}/id_cv_deep_preds_3seeds.csv")
m = ID3[ID3.seed == 42][["filename", "vendor", "stage", "fold"]].reset_index(drop=True)
F7 = pd.read_csv(f"{CKPT}/lifs_stad_fixed.csv"); cols = [c for c in F7.columns if c not in ("filename", "vendor", "stage")]
X = F7.set_index("filename").loc[m.filename, cols].to_numpy(dtype=float)
rf = lambda: make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num),
                           RandomForestClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                                                  class_weight="balanced", n_jobs=-1, random_state=42))
for t, thr in [("sub", 2), ("cir", 4)]:
    yy = (m.stage.to_numpy() >= thr).astype(int); oof = np.zeros(len(m))
    for k in sorted(m.fold.unique()):
        te = m.fold.to_numpy() == k; oof[te] = rf().fit(X[~te], yy[~te]).predict_proba(X[te])[:, 1]
    m[f"rf_{t}"] = oof
d3 = ID3.groupby("filename")[["d1", "d4"]].mean(); d42 = ID3[ID3.seed == 42].set_index("filename")[["d1", "d4"]]
m["deep3_sub"], m["deep3_cir"] = 1 - d3.loc[m.filename, "d1"].to_numpy(), d3.loc[m.filename, "d4"].to_numpy()
m["deep42_sub"], m["deep42_cir"] = 1 - d42.loc[m.filename, "d1"].to_numpy(), d42.loc[m.filename, "d4"].to_numpy()
for t in ["sub", "cir"]:
    m[f"fus3_{t}"] = 0.5 * m[f"deep3_{t}"] + 0.5 * m[f"rf_{t}"]

# (b) LOSO AUC per scanner for the same deep stream (3 seeds) and forest, as in Table 4
L = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv").groupby(["filename", "held", "stage"], as_index=False)[["d1", "d4"]].mean()
BC = pd.read_csv(f"{RES}/B_loco_cases.csv"); BC = BC[(BC.feat == "fixed7") & (BC.variant == "raw")][["filename", "rf_sub", "rf_cir"]]
L = L.merge(BC, on="filename")
rows = []
for scope in ["pooled"] + CENTERS:
    g = m if scope == "pooled" else m[m.vendor.astype(str) == scope]
    for t, thr in [("sub", 2), ("cir", 4)]:
        yy = (g.stage.to_numpy() >= thr).astype(int)
        rec = dict(scope=scope, task=t, n_neg=int((yy == 0).sum()), **{f"ID_{k}": roc_auc_score(yy, g[f"{k}_{t}"]) for k in ["deep3", "deep42", "rf", "fus3"]})
        if scope != "pooled":
            l = L[L.held == scope]; yl = (l.stage.to_numpy() >= thr).astype(int)
            rec["LOSO_deep3"] = roc_auc_score(yl, (1 - l.d1) if t == "sub" else l.d4)
            rec["LOSO_rf"] = roc_auc_score(yl, l[f"rf_{t}"])
            rec["deep_change"], rec["rf_change"] = rec["LOSO_deep3"] - rec["ID_deep3"], rec["LOSO_rf"] - rec["ID_rf"]
        rows.append(rec)
T = pd.DataFrame(rows); pd.set_option("display.width", 250)
print("=== (a, b) in-distribution vs LOSO AUC (task sub = S2+, cir = S4) ===\n" + T.round(3).to_string(index=False))
T.to_csv(f"{RES}/C2_auc_id_vs_loso.csv", index=False)

# (c) pooled C-index with 95% CI for the three interventions (same bootstrap draws as CELL V), for Table 3
D = L[["filename", "held", "stage"]].copy(); D["deep"] = (1 - L.d1) + L.d4
for mth in ["multiphase", "intensity_aug", "adversarial"]:
    g = pd.read_csv(f"{DG}/dg_preds_{mth}.csv").groupby("filename", as_index=False)[["d1", "d4"]].mean()
    D = D.merge(pd.DataFrame({"filename": g.filename, mth: (1 - g.d1) + g.d4}), on="filename", how="left")
D = D.sort_values("filename").reset_index(drop=True)
MS = ["deep", "multiphase", "intensity_aug", "adversarial"]
A = {h: (D[D.held == h].stage.to_numpy(dtype=int), {k: D[D.held == h][k].to_numpy() for k in MS}) for h in CENTERS}
def pooled(idx):
    c = {k: 0.0 for k in MS}; n = 0
    for h in CENTERS:
        st = A[h][0][idx[h]]; comp = st[:, None] > st[None, :]; n += comp.sum()
        for k in MS:
            x = A[h][1][k][idx[h]]; dx = (x[:, None] - x[None, :])[comp]; c[k] += (dx > 0).sum() + 0.5 * (dx == 0).sum()
    return np.array([c[k] / n for k in MS])
r = np.random.default_rng(0)
W = [{h: r.integers(0, len(A[h][0]), len(A[h][0])) for h in CENTERS} for _ in range(2000)]
pt = pooled({h: np.arange(len(A[h][0])) for h in CENTERS}); bt = np.array([pooled(w) for w in W])
IC = pd.DataFrame([dict(model=k, C=pt[i], ci=f"{np.percentile(bt[:, i], 2.5):.3f}-{np.percentile(bt[:, i], 97.5):.3f}") for i, k in enumerate(MS)])
print("\n=== (c) pooled within-scanner C-index, 95% CI (deep must read 0.646, 0.602-0.688) ===\n" + IC.round(3).to_string(index=False))
IC.to_csv(f"{RES}/C1_intervention_C_ci.csv", index=False)

# (d) one ledger file: every summary table in paper_results, stacked, with its source file
parts = []
for p in sorted(glob.glob(f"{RES}/*.csv")):
    df = pd.read_csv(p)
    if len(df) > 400: continue                                  # case-level prediction files are inputs, not reported numbers
    df = df.astype(str); df.insert(0, "source_file", os.path.basename(p)); parts.append(df)
pd.concat(parts, ignore_index=True).to_csv(f"{RES}/LEDGER_all_results.csv", index=False)
print(f"\nledger: {len(parts)} result tables stacked into LEDGER_all_results.csv")